In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

project_root = Path.cwd()
if project_root.name == "notebooks":
    project_root = project_root.parent

previous_cache = project_root / "data/processed/lab10"
cache_dir = project_root / "data/processed/lab11"
cache_dir.mkdir(parents=True, exist_ok=True)

train_pool = pd.read_parquet(
    previous_cache / "corpus.parquet"
)
dev_queries = pd.read_parquet(
    previous_cache / "queries.parquet"
)

assert train_pool["question_id"].is_unique
assert dev_queries["question_id"].is_unique

assert set(train_pool["question_id"]).isdisjoint(
    set(dev_queries["question_id"])
)
assert set(train_pool["text_key"]).isdisjoint(
    set(dev_queries["text_key"])
)

print(f"Training/reference questions: {len(train_pool):,}")
print(f"Development queries: {len(dev_queries):,}")

Training/reference questions: 1,270
Development queries: 318


In [2]:
def make_training_triplets(
    frame: pd.DataFrame,
    *,
    seed: int = 42,
) -> pd.DataFrame:
    """Create one category-based triplet per question."""
    groups = {
        category: group["question_id"].to_numpy()
        for category, group in frame.groupby(
            "category", sort=True
        )
    }

    if (
        len(groups) < 2
        or any(len(ids) < 2 for ids in groups.values())
    ):
        raise ValueError(
            "Need at least two categories "
            "and two questions per category."
        )

    rng = np.random.default_rng(seed)
    records = []

    for row in frame.itertuples(index=False):
        positive_ids = groups[row.category]
        positive_ids = positive_ids[
            positive_ids != row.question_id
        ]

        other_categories = [
            category
            for category in groups
            if category != row.category
        ]
        negative_category = rng.choice(other_categories)

        records.append({
            "anchor_id": int(row.question_id),
            "positive_id": int(rng.choice(positive_ids)),
            "negative_id": int(
                rng.choice(groups[negative_category])
            ),
        })

    return pd.DataFrame(records)


triplets = make_training_triplets(train_pool)

print(f"Training triplets: {len(triplets):,}")
display(triplets.head())

Training triplets: 1,270


,anchor_id,positive_id,negative_id
0,2794,2796,8649
1,5640,5593,7439
2,2167,2119,8674
3,6346,6459,8774
4,6428,6357,6023


In [3]:
lookup = train_pool.set_index("question_id")

for role in ["anchor", "positive", "negative"]:
    assert triplets[f"{role}_id"].isin(lookup.index).all()

assert triplets["anchor_id"].is_unique
assert triplets["anchor_id"].ne(
    triplets["positive_id"]
).all()

anchor_categories = triplets["anchor_id"].map(
    lookup["category"]
)
positive_categories = triplets["positive_id"].map(
    lookup["category"]
)
negative_categories = triplets["negative_id"].map(
    lookup["category"]
)

assert anchor_categories.eq(positive_categories).all()
assert anchor_categories.ne(negative_categories).all()

print("Triplet checks passed.")

Triplet checks passed.


In [6]:
inspection = triplets.head(3).copy()
inspection

,anchor_id,positive_id,negative_id
0,2794,2796,8649
1,5640,5593,7439
2,2167,2119,8674


In [8]:
inspection = triplets.head(3).copy()

for role in ["anchor", "positive", "negative"]:
    inspection[f"{role}_text"] = inspection[
        f"{role}_id"
    ].map(lookup["text"])

    inspection[f"{role}_category"] = inspection[
        f"{role}_id"
    ].map(lookup["category"])

for row in inspection.itertuples(index=False):
    print("ANCHOR:", row.anchor_text)
    print("Category:", row.anchor_category)

    print("POSITIVE:", row.positive_text)
    print("Category:", row.positive_category)

    print("NEGATIVE:", row.negative_text)
    print("Category:", row.negative_category)
    print()

ANCHOR: How long will my transaction take?
Category: transfer_not_received_by_recipient
POSITIVE: i accidentally entered in my retailers information, what should i do next
Category: transfer_not_received_by_recipient
NEGATIVE: Where is my money that I transfered?!
Category: balance_not_updated_after_bank_transfer

ANCHOR: I can't seem to figure out why my transfers are getting declined.
Category: declined_transfer
POSITIVE: What does the decline message mean
Category: declined_transfer
NEGATIVE: What would cause a transfer failure?
Category: failed_transfer

ANCHOR: I need to cancel my recent transfer as soon as possible. I made an error there. Please help before it goes through.
Category: cancel_transfer
POSITIVE: I wanna revert a transaction i did this morning.
Category: cancel_transfer
NEGATIVE: I transfered my balance a while back.  Why doesn't my account reflect this?
Category: balance_not_updated_after_bank_transfer



In [10]:
train_pool.to_parquet(
    cache_dir / "train_pool.parquet",
    index=False,
)
dev_queries.to_parquet(
    cache_dir / "dev_queries.parquet",
    index=False,
)
triplets.to_parquet(
    cache_dir / "training_triplets.parquet",
    index=False,
)

print("Lab 11 training examples saved.")

Lab 11 training examples saved.


# Step 2 — Fine-tune and compare on development queries

In [12]:
import json
from time import perf_counter

import torch
from torch.nn import functional as F
from sentence_transformers import SentenceTransformer

torch.set_num_threads(2)

config = {
    "model_id": "sentence-transformers/all-MiniLM-L6-v2",
    "seed": 42,
    "epochs": 2,
    "batch_size": 8,
    "margin": 0.2,
    "learning_rate": 2e-5,
    "weight_decay": 0.01,
}

text_lookup = train_pool.set_index("question_id")["text"]

text_triplets = pd.DataFrame({
    role: triplets[f"{role}_id"].map(text_lookup)
    for role in ["anchor", "positive", "negative"]
})

assert text_triplets.notna().all().all()

/Users/sara/Projects/data-science-lab/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
W1122 16:06:38.103000 44472 torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.


In [15]:
text_triplets

,anchor,positive,negative
0,How long will my transaction take?,i accidentally entered in my retailers informa...,Where is my money that I transfered?!
1,I can't seem to figure out why my transfers ar...,What does the decline message mean,What would cause a transfer failure?
2,I need to cancel my recent transfer as soon as...,I wanna revert a transaction i did this morning.,I transfered my balance a while back. Why doe...
3,Transaction showing up many times.,Looks like my card payment was made twice by m...,my transfer isn't there
4,I have a transaction showing up multiple times.,"Good day, in reviewing my credit card activity...",As I double checked that I have the correct ac...
...,...,...,...
1265,Tell me why my balance did not change when I t...,"I recently transferred some cash, but the bala...","I wanted to buy something online yesterday, bu..."
1266,I have tried a couple of times to make a purch...,Can you tell me why my transfer declined?,I don't see any change in my balance.
1267,I need a refund for something I bought.,How can I go about getting a refund?,Why is my card payment still pending? It shoul...
1268,Why didn't my friend's transaction arrive yet?,|I moved out of my old house two weeks ago and...,My balance doesn't include my latest transfer


In [17]:
def train_triplet_epoch(
    model,
    text_triplets,
    optimizer,
    *,
    rng,
    batch_size=8,
    margin=0.2,
):
    model.train()

    order = rng.permutation(len(text_triplets))
    loss_total = 0.0
    active_total = 0

    for start in range(0, len(order), batch_size):
        batch = text_triplets.iloc[
            order[start:start + batch_size]
        ]

        texts = (
            batch["anchor"].tolist()
            + batch["positive"].tolist()
            + batch["negative"].tolist()
        )

        optimizer.zero_grad(set_to_none=True)

        features = model.preprocess(texts)
        output = model(features)

        vectors = F.normalize(
            output["sentence_embedding"],
            p=2,
            dim=1,
        )
        anchor, positive, negative = vectors.chunk(3, dim=0)

        positive_similarity = (anchor * positive).sum(dim=1)
        negative_similarity = (anchor * negative).sum(dim=1)

        losses = F.relu(
            margin + negative_similarity - positive_similarity
        )
        loss = losses.mean()

        if not torch.isfinite(loss).item():
            raise FloatingPointError("Training loss is not finite.")

        loss.backward()
        optimizer.step()

        loss_total += loss.item() * len(batch)
        active_total += int(
            (losses.detach() > 0).sum().item()
        )

        step = start // batch_size + 1
        if step % 40 == 0:
            print(f"  Finished {step} batches", flush=True)

    return {
        "train_loss": loss_total / len(order),
        "active_triplet_fraction": active_total / len(order),
    }

In [18]:
def evaluate_encoder(model, corpus, queries):
    encode_options = {
        "batch_size": 32,
        "normalize_embeddings": True,
        "convert_to_numpy": True,
        "show_progress_bar": False,
    }

    corpus_vectors = model.encode(
        corpus["text"].tolist(),
        **encode_options,
    )
    query_vectors = model.encode(
        queries["text"].tolist(),
        **encode_options,
    )

    scores = query_vectors @ corpus_vectors.T

    top_positions = np.argsort(
        -scores,
        axis=1,
        kind="stable",
    )[:, :5]

    matches = (
        corpus["category"].to_numpy()[top_positions]
        == queries["category"].to_numpy()[:, None]
    )

    reciprocal_ranks = (
        matches / np.arange(1, 6)
    ).max(axis=1)

    metrics = {
        "top1_accuracy": float(matches[:, 0].mean()),
        "hit_at_5": float(matches.any(axis=1).mean()),
        "mrr_at_5": float(reciprocal_ranks.mean()),
    }

    return metrics, scores

In [19]:
torch.manual_seed(config["seed"])

model = SentenceTransformer(
    config["model_id"],
    device="cpu",
    local_files_only=True,
)

np.testing.assert_array_equal(
    train_pool["question_id"].to_numpy(),
    np.load(previous_cache / "corpus_question_ids.npy"),
)
np.testing.assert_array_equal(
    dev_queries["question_id"].to_numpy(),
    np.load(previous_cache / "query_question_ids.npy"),
)

start = perf_counter()
baseline_metrics, baseline_scores = evaluate_encoder(
    model, train_pool, dev_queries
)
baseline_eval_seconds = perf_counter() - start

np.testing.assert_allclose(
    baseline_scores,
    np.load(previous_cache / "embedding_scores.npy"),
    rtol=1e-5,
    atol=1e-6,
)

print("Baseline reproduced:", baseline_metrics)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 8017.47it/s]


Baseline reproduced: {'top1_accuracy': 0.9088050314465409, 'hit_at_5': 0.9685534591194969, 'mrr_at_5': 0.934748427672956}


In [20]:
models_dir = project_root / "models"
reports_dir = project_root / "reports"

models_dir.mkdir(parents=True, exist_ok=True)
reports_dir.mkdir(parents=True, exist_ok=True)

model.save_pretrained(
    str(models_dir / "lab11_original_encoder"),
    create_model_card=False,
)

records = [{
    "epoch": 0,
    "train_loss": np.nan,
    "active_triplet_fraction": np.nan,
    "train_seconds": 0.0,
    "eval_seconds": baseline_eval_seconds,
    **baseline_metrics,
}]

Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  6.12it/s]


In [21]:
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=config["learning_rate"],
    weight_decay=config["weight_decay"],
)
rng = np.random.default_rng(config["seed"])

for epoch in range(1, config["epochs"] + 1):
    print(f"Starting epoch {epoch}/{config['epochs']}", flush=True)

    start = perf_counter()
    training_metrics = train_triplet_epoch(
        model,
        text_triplets,
        optimizer,
        rng=rng,
        batch_size=config["batch_size"],
        margin=config["margin"],
    )
    train_seconds = perf_counter() - start

    start = perf_counter()
    dev_metrics, dev_scores = evaluate_encoder(
        model, train_pool, dev_queries
    )
    eval_seconds = perf_counter() - start

    records.append({
        "epoch": epoch,
        **training_metrics,
        "train_seconds": train_seconds,
        "eval_seconds": eval_seconds,
        **dev_metrics,
    })

    print(
        f"Epoch {epoch} | "
        f"loss={training_metrics['train_loss']:.4f} | "
        f"top-1={dev_metrics['top1_accuracy']:.2%} | "
        f"MRR@5={dev_metrics['mrr_at_5']:.4f} | "
        f"training={train_seconds:.1f}s",
        flush=True,
    )

    pd.DataFrame(records).to_csv(
        reports_dir / "lab_11_development.csv",
        index=False,
    )

development_results = pd.DataFrame(records)
display(development_results.round(4))

Starting epoch 1/2
  Finished 40 batches
  Finished 80 batches
  Finished 120 batches
Epoch 1 | loss=0.0360 | top-1=91.51% | MRR@5=0.9351 | training=59.8s
Starting epoch 2/2
  Finished 40 batches
  Finished 80 batches
  Finished 120 batches
Epoch 2 | loss=0.0127 | top-1=92.45% | MRR@5=0.9382 | training=51.7s


,epoch,train_loss,active_triplet_fraction,train_seconds,eval_seconds,top1_accuracy,hit_at_5,mrr_at_5
0,0,NaN,NaN,0.0000,2.1005,0.9088,0.9686,0.9347
1,1,0.0360,0.2433,59.8436,1.9572,0.9151,0.9623,0.9351
2,2,0.0127,0.0945,51.6638,1.8437,0.9245,0.9591,0.9382


In [22]:
model.save_pretrained(
    str(models_dir / "lab11_finetuned_encoder"),
    create_model_card=False,
)

np.save(cache_dir / "baseline_dev_scores.npy", baseline_scores)
np.save(cache_dir / "finetuned_dev_scores.npy", dev_scores)

np.save(
    cache_dir / "corpus_question_ids.npy",
    train_pool["question_id"].to_numpy(),
)
np.save(
    cache_dir / "dev_question_ids.npy",
    dev_queries["question_id"].to_numpy(),
)

(cache_dir / "training_config.json").write_text(
    json.dumps(config, indent=2) + "\n"
)

Writing model shards: 100%|██████████| 1/1 [00:01<00:00,  1.12s/it]


174

# Step 3 —  Compare the original and adapted models on separate final queries, then write the report.

In [23]:
from urllib.request import urlretrieve

raw_dir = project_root / "data/raw/banking77"
raw_dir.mkdir(parents=True, exist_ok=True)

test_path = raw_dir / "test.csv"
test_url = (
    "https://raw.githubusercontent.com/"
    "PolyAI-LDN/task-specific-datasets/"
    "master/banking_data/test.csv"
)

if not test_path.exists():
    urlretrieve(test_url, test_path)

raw_test = pd.read_csv(test_path)

# Prefix IDs to distinguish them from the training-file IDs.
raw_test.insert(
    0,
    "question_id",
    [f"test_{i}" for i in range(len(raw_test))],
)

assert raw_test[["text", "category"]].notna().all().all()

final_queries = raw_test.loc[
    raw_test["category"].isin(train_pool["category"].unique())
].copy()

selected_count = len(final_queries)

In [24]:
final_queries["text_key"] = (
    final_queries["text"]
    .str.casefold()
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

assert final_queries["text_key"].ne("").all()

conflicting = (
    final_queries.groupby("text_key")["category"]
    .transform("nunique")
    .gt(1)
)
conflict_count = int(conflicting.sum())

final_queries = final_queries.loc[~conflicting].copy()

before = len(final_queries)
final_queries = final_queries.drop_duplicates("text_key")
duplicate_count = before - len(final_queries)

known_texts = (
    set(train_pool["text_key"])
    | set(dev_queries["text_key"])
)

overlap = final_queries["text_key"].isin(known_texts)
overlap_count = int(overlap.sum())

final_queries = (
    final_queries.loc[~overlap]
    .reset_index(drop=True)
)

assert len(final_queries) > 0
assert final_queries["question_id"].is_unique
assert final_queries["text_key"].is_unique
assert set(final_queries["text_key"]).isdisjoint(known_texts)
assert set(final_queries["category"]) == set(train_pool["category"])

print(f"Selected questions: {selected_count}")
print(f"Conflicting-label rows excluded: {conflict_count}")
print(f"Repeated-text rows excluded: {duplicate_count}")
print(f"Training/development overlaps excluded: {overlap_count}")
print(f"Final evaluation queries: {len(final_queries)}")

Selected questions: 400
Conflicting-label rows excluded: 0
Repeated-text rows excluded: 0
Training/development overlaps excluded: 0
Final evaluation queries: 400


In [25]:
final_queries.to_parquet(
    cache_dir / "final_queries.parquet",
    index=False,
)

np.save(
    cache_dir / "final_query_ids.npy",
    final_queries["question_id"].to_numpy(dtype=str),
)

In [26]:
np.testing.assert_array_equal(
    train_pool["question_id"].to_numpy(),
    np.load(cache_dir / "corpus_question_ids.npy"),
)

test_records = []

for name, directory in [
    ("original", "lab11_original_encoder"),
    ("fine_tuned", "lab11_finetuned_encoder"),
]:
    test_encoder = SentenceTransformer(
        str(models_dir / directory),
        device="cpu",
        local_files_only=True,
    )

    metrics, scores = evaluate_encoder(
        test_encoder,
        train_pool,
        final_queries,
    )

    assert scores.shape == (
        len(final_queries),
        len(train_pool),
    )
    assert np.isfinite(scores).all()

    test_records.append({
        "model": name,
        "query_count": len(final_queries),
        **metrics,
    })

    np.save(
        cache_dir / f"{name}_test_scores.npy",
        scores,
    )

test_comparison = pd.DataFrame(test_records)

display(test_comparison.round(4))

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 25704.37it/s]


,model,query_count,top1_accuracy,hit_at_5,mrr_at_5
0,original,400,0.9050,0.9675,0.9289
1,fine_tuned,400,0.9125,0.9625,0.9328


In [27]:
test_comparison.to_csv(
    reports_dir / "lab_11_test_comparison.csv",
    index=False,
)